<a href="https://colab.research.google.com/github/hwasun-zip/yamyam-web/blob/main/feedback-analysis/feedback_classify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# yamyam 피드백 자동 분류 (LLM)

## 왜 만들었나
yamyam-web은 배포할 때마다 설문 서술형 의견을 **직접 읽고 주제별로 묶어** 다음 개선의 우선순위를 정했다 (v1 18건, v2 8건).
이 방식은 두 가지 문제가 있었다.

1. **의견이 늘수록 시간이 선형으로 늘어난다.** 버전마다 같은 작업을 처음부터 반복한다.
2. **분류 기준이 흔들린다.** v1과 v2의 주제 목록이 달라 버전 간 비교가 어렵고, 한 의견에 여러 요청이 섞이면 놓치기 쉽다.

## 무엇을 하나
1. 고정된 주제 정의(taxonomy)를 주고 Claude가 의견마다 주제를 여러 개 고르게 한다 (근거 구절 포함).
2. 내가 직접 한 분류(정답)와 비교해 **믿고 쓸 수 있는 수준인지** 검증한다. LLM에는 정답을 보여주지 않는다.
3. 버전별 **우선순위 표를 자동으로 생성**한다. 다음 버전 설문이 쌓이면 CSV만 바꿔서 다시 실행하면 된다.

## 실행 전 준비
- 코랩 왼쪽 🔑(Secrets)에 `ANTHROPIC_API_KEY` 등록 (없으면 실행 중 입력 창이 뜬다)
- Google Forms 응답 CSV 파일


In [1]:
!pip install -q anthropic tabulate
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API 키: ")

MODEL = "claude-sonnet-5-5"


In [2]:
from google.colab import files
uploaded = files.upload()          # Google Forms 응답 CSV 선택
fname = list(uploaded)[0]


Saving 제목 없는 설문지.csv.zip to 제목 없는 설문지.csv.zip


In [3]:
import pandas as pd
from datetime import datetime

raw = pd.read_csv(fname, encoding="utf-8-sig", dtype=str).fillna("")
raw.columns = ["ts_raw", "satisfaction", "visit", "features", "text", "reuse"]

def parse_ts(s):
    # 예: "2026/10/23 10:40:47 오전 GMT+9"
    s = s.replace(" GMT+9", "").replace("오전", "AM").replace("오후", "PM").strip()
    return datetime.strptime(s, "%Y/%m/%d %I:%M:%S %p")

V2_RELEASE = datetime(2026, 10, 23, 22, 44)   # v2 배포 시각
raw["ts"] = raw["ts_raw"].map(parse_ts)
raw["version"] = raw["ts"].map(lambda t: "v1" if t < V2_RELEASE else "v2")
raw["key"] = raw["ts"].dt.strftime("%Y-%m-%d %H:%M:%S")
raw["text"] = raw["text"].str.strip()

df = raw[raw["text"] != ""].reset_index(drop=True)
print("전체 응답:", len(raw), "/ 서술형 있는 응답:", len(df))
print(df["version"].value_counts().sort_index().to_string())


전체 응답: 30 / 서술형 있는 응답: 26
version
v1    18
v2     8


## 주제 정의와 정답(사람 분류)

- 주제는 v1·v2 README에서 실제로 쓴 주제를 하나의 목록으로 합쳤다.
- **정답은 README의 주제별 합계에 맞춰 응답 단위로 복원했다.** v2 8건은 합계와 완전히 일치한다.
  v1에서 '지역 확대' 8번째 1건은 기록이 남아 있지 않아, v2에서 "내 위치 주소 표시"를 지역 문제의 해결책으로 넣은 점을 근거로 10/23 12:28 응답에 배정했다.
- 사람 분류 당시 v1에는 `PHOTO`, `SCORE_SCALE`, `PRAISE_FIX` 주제가 없었다. 그래서 v1에서 LLM이 이 주제를 고르면 "사람이 당시 쓰지 않은 주제"로 따로 표시한다.


In [4]:
THEMES = {
    "REGION":       ("지역 확대",          "강남구 밖 지역이나 내 동네 지원 요청. 현재 위치 주소 표시 요청 포함"),
    "MENU":         ("음식 종류·메뉴 수",   "추천 음식 종류를 늘리거나, 전체 메뉴와 점수를 보고 싶다는 요청"),
    "PERSONAL":     ("개인 취향·회원",      "취향, 기분, 사용자 정보, 회원 기능을 반영한 맞춤 추천 요청"),
    "MAP":          ("지도",               "다른 지도 앱(네이버, 구글) 연동, 페이지 안 지도, 지도로 이동하는 방식에 대한 요청"),
    "REASON":       ("추천 이유·점수 의미", "왜 이 음식을 추천했는지, 점수(+0.6 등)가 무슨 뜻인지 설명 요청"),
    "CATEGORY":     ("분류·정렬",          "한식/일식, 국물/밥 같은 분류나 음식별 정렬·필터 요청"),
    "REVIEW":       ("리뷰",               "사용자가 리뷰를 남기거나 볼 수 있는 기능 요청"),
    "LOCATION_BUG": ("위치 오류",          "위치 권한, 위치 확인 버튼 등 위치 기능의 오류 제보"),
    "PHOTO":        ("사진·아이콘",        "음식 사진, 썸네일, 음식 아이콘 개선 요청"),
    "SCORE_SCALE":  ("점수 표현",          "점수 숫자의 크기나 표현 방식이 와닿지 않는다는 의견"),
    "PRAISE_FIX":   ("개선 호평",          "이전 버전에서 고쳐진 기능(지역, 지도 앱, 추천 이유 등)이 좋았다는 의견"),
}
V1_ONLY_THEMES = {"REGION","MENU","PERSONAL","MAP","REASON","CATEGORY","REVIEW","LOCATION_BUG"}

# 사람 분류(정답). 키 = 응답 시각
GOLD = {
    # v1
    "2026-10-23 10:40:47": ["REGION"],
    "2026-10-23 10:56:01": ["PERSONAL", "REVIEW"],
    "2026-10-23 11:18:05": ["MAP"],
    "2026-10-23 11:21:35": [],
    "2026-10-23 11:26:46": [],
    "2026-10-23 11:39:34": ["REGION"],
    "2026-10-23 11:50:01": ["CATEGORY", "MAP"],
    "2026-10-23 12:16:20": ["REVIEW"],
    "2026-10-23 12:26:52": ["REGION", "PERSONAL"],
    "2026-10-23 12:28:10": ["PERSONAL", "MENU", "REGION", "LOCATION_BUG", "MAP"],
    "2026-10-23 12:37:30": ["REASON"],
    "2026-10-23 12:56:13": ["MENU"],
    "2026-10-23 13:02:06": ["REGION"],
    "2026-10-23 13:12:49": ["REGION", "CATEGORY"],
    "2026-10-23 16:51:03": ["REASON", "MENU"],
    "2026-10-23 19:23:29": ["REGION"],
    "2026-10-23 22:12:01": ["MENU", "REGION", "REASON", "MAP"],
    "2026-10-23 22:18:31": ["MENU", "CATEGORY", "PERSONAL"],
    # v2
    "2026-10-24 12:20:34": ["PRAISE_FIX", "MENU"],
    "2026-10-24 13:51:34": ["PRAISE_FIX"],
    "2026-10-24 13:57:07": ["PRAISE_FIX", "MENU", "CATEGORY"],
    "2026-10-24 13:59:41": ["PRAISE_FIX"],
    "2026-10-24 14:43:23": ["PRAISE_FIX", "MENU", "REGION"],
    "2026-10-24 15:15:40": ["MENU"],
    "2026-10-24 16:18:06": ["MENU", "CATEGORY", "PHOTO"],
    "2026-10-25 02:30:57": ["MENU", "SCORE_SCALE", "PHOTO"],
}
missing = set(df["key"]) - set(GOLD)
assert not missing, f"정답이 없는 응답: {missing}"
df["human"] = df["key"].map(lambda k: sorted(GOLD[k]))
print("정답 준비 완료:", len(df), "건")


정답 준비 완료: 26 건


In [11]:
import json, re, anthropic
client = anthropic.Anthropic()

theme_lines = "\n".join(f"- {code}: {name} — {desc}" for code, (name, desc) in THEMES.items())
SYSTEM = f"""너는 서비스 사용자의 서술형 의견을 주제별로 분류하는 분석가다.

[주제 목록]
{theme_lines}

[규칙]
- 위 목록의 코드만 사용한다. 한 의견에 여러 요청이 있으면 해당하는 주제를 모두 고른다.
- 의견 안에 근거 문장이 있는 주제만 고른다. 추측하지 않는다.
- 목록에 맞지 않는 요청이 있으면 other에 한 줄로 요약한다. 없으면 null.
- 설명 없이 아래 JSON 하나만 출력한다.
{{"labels": ["코드", ...], "evidence": {{"코드": "의견에서 그대로 가져온 짧은 구절"}}, "other": null}}"""

decoder = json.JSONDecoder()

def extract_json(out):
    """답변에서 'labels'가 들어 있는 첫 번째 JSON 객체만 꺼낸다."""
    for m in re.finditer(r"\{", out):
        try:
            obj, _ = decoder.raw_decode(out[m.start():])
            if isinstance(obj, dict) and "labels" in obj:
                return obj
        except json.JSONDecodeError:
            continue
    return None

def classify(text):
    resp = client.messages.create(
        model=MODEL, max_tokens=4000, system=SYSTEM,
        messages=[{"role": "user", "content": f"[의견]\n{text}"}],
    )
    out = "".join(b.text for b in resp.content if b.type == "text")   # 생각(thinking) 블록은 건너뜀
    data = extract_json(out) or {"labels": [], "evidence": {}, "other": f"파싱 실패: {out[:80]}"}
    data["labels"] = sorted({l for l in data.get("labels", []) if l in THEMES})
    data.setdefault("evidence", {})
    return data

# 동작 확인 1건
print(classify(df.loc[0, "text"]))

{'labels': ['REGION'], 'evidence': {'REGION': '강남구 말고 더 넓은 지역도 됐으면 좋겠어요'}, 'other': None}


In [14]:
import time
t0 = time.time()
results = [classify(t) for t in df["text"]]
elapsed = time.time() - t0

df["llm"] = [r["labels"] for r in results]
df["evidence"] = [r.get("evidence", {}) for r in results]
df["other"] = [r.get("other") for r in results]
print(f"{len(df)}건 분류 완료: {elapsed:.1f}초 (건당 {elapsed/len(df):.1f}초)")

26건 분류 완료: 76.9초 (건당 3.0초)


## 검증: 사람 분류와 얼마나 같은가

In [15]:
def jaccard(a, b):
    a, b = set(a), set(b)
    return 1.0 if not a and not b else len(a & b) / len(a | b)

df["exact"] = [set(h) == set(l) for h, l in zip(df["human"], df["llm"])]
df["jaccard"] = [jaccard(h, l) for h, l in zip(df["human"], df["llm"])]

print(f"응답 단위 완전 일치: {df['exact'].mean():.0%} ({df['exact'].sum()}/{len(df)})")
print(f"응답 단위 평균 겹침(Jaccard): {df['jaccard'].mean():.2f}")

rows = []
for code, (name, _) in THEMES.items():
    tp = sum(code in h and code in l for h, l in zip(df["human"], df["llm"]))
    fp = sum(code not in h and code in l for h, l in zip(df["human"], df["llm"]))
    fn = sum(code in h and code not in l for h, l in zip(df["human"], df["llm"]))
    if tp + fp + fn == 0:
        continue
    rows.append({"주제": name, "사람": tp + fn, "LLM": tp + fp, "일치": tp,
                 "정밀도": tp / (tp + fp) if tp + fp else None,
                 "재현율": tp / (tp + fn) if tp + fn else None})
per_theme = pd.DataFrame(rows)
tp_all, fp_all, fn_all = per_theme["일치"].sum(), (per_theme["LLM"] - per_theme["일치"]).sum(), (per_theme["사람"] - per_theme["일치"]).sum()
print(f"전체 정밀도 {tp_all/(tp_all+fp_all):.0%} · 재현율 {tp_all/(tp_all+fn_all):.0%}")
per_theme.style.format({"정밀도": "{:.0%}", "재현율": "{:.0%}"}, na_rep="-")


응답 단위 완전 일치: 92% (24/26)
응답 단위 평균 겹침(Jaccard): 0.97
전체 정밀도 96% · 재현율 100%


,주제,사람,LLM,일치,정밀도,재현율
0,지역 확대,9,9,9,100%,100%
1,음식 종류·메뉴 수,11,11,11,100%,100%
2,개인 취향·회원,4,5,4,80%,100%
3,지도,4,4,4,100%,100%
4,추천 이유·점수 의미,3,3,3,100%,100%
5,분류·정렬,5,5,5,100%,100%
6,리뷰,2,2,2,100%,100%
7,위치 오류,1,1,1,100%,100%
8,사진·아이콘,2,3,2,67%,100%
9,점수 표현,1,1,1,100%,100%


## 불일치 검토
LLM과 사람의 분류가 다른 응답만 모았다. **LLM이 틀린 것인지, 사람이 놓친 것인지** 직접 판정한다.
v1에서 당시 없던 주제(사진, 점수 표현, 개선 호평)를 LLM이 고른 경우는 `당시없던주제`로 표시했다.


In [16]:
diff_rows = []
for _, r in df[~df["exact"]].iterrows():
    h, l = set(r["human"]), set(r["llm"])
    only_llm, only_human = sorted(l - h), sorted(h - l)
    note = ""
    if r["version"] == "v1" and only_llm and all(c not in V1_ONLY_THEMES for c in only_llm):
        note = "당시없던주제"
    diff_rows.append({
        "버전": r["version"], "시각": r["key"][5:16],
        "의견(앞부분)": r["text"][:60].replace("\n", " ") + ("…" if len(r["text"]) > 60 else ""),
        "LLM만": ", ".join(only_llm), "사람만": ", ".join(only_human),
        "LLM 근거": "; ".join(f"{k}: {v}" for k, v in r["evidence"].items() if k in only_llm),
        "비고": note,
    })
diff = pd.DataFrame(diff_rows)
print("불일치", len(diff), "건")
diff


불일치 2 건


,버전,시각,의견(앞부분),LLM만,사람만,LLM 근거,비고
0,v1,10-23 11:18,"음식점 배너 배열이 2열이고, 지도버튼을 눌러야 넘어가는게 아니라 배너 전체를 눌러...",PHOTO,,PHOTO: 음식점 리뷰 사진을 긁어모아서 썸네일 처럼 사진이 있었으면 좋겠다,당시없던주제
1,v2,10-24 14:43,음식 종류가 더 다양해지면 좋겠습니다! 동네도 늘어나면 좋을 것 같아요! 사용자 정...,PERSONAL,,PERSONAL: 사용자 정보가 추가되면 좋을 것 같아요!,


## 자동 생성: 버전별 우선순위 표
이 표가 수작업을 대체하는 결과물이다. 다음 버전 설문이 쌓이면 CSV만 바꿔 다시 실행하면 된다.


In [17]:
def priority_table(version):
    sub = df[df["version"] == version]
    n = len(sub)
    rows = []
    for code, (name, _) in THEMES.items():
        hit = sub[sub["llm"].map(lambda ls: code in ls)]
        if len(hit) == 0:
            continue
        example = next((e.get(code) for e in hit["evidence"] if e.get(code)), "")
        rows.append({"주제": name, "건수": len(hit), "비중": f"{len(hit)/n:.0%}", "대표 의견": example})
    return pd.DataFrame(rows).sort_values("건수", ascending=False).reset_index(drop=True)

for v in sorted(df["version"].unique()):
    n = (df["version"] == v).sum()
    print(f"\n### {v} (서술형 {n}건)")
    print(priority_table(v).to_markdown(index=False))

others = df[df["other"].notna() & (df["other"] != "null")][["version", "other"]]
if len(others):
    print("\n[목록에 없는 요청 — 새 주제 후보]")
    print(others.to_string(index=False))



### v1 (서술형 18건)
| 주제                |   건수 | 비중   | 대표 의견                                                                                                |
|:--------------------|-------:|:-------|:---------------------------------------------------------------------------------------------------------|
| 지역 확대           |      8 | 44%    | 강남구 말고 더 넓은 지역도 됐으면 좋겠어요                                                               |
| 음식 종류·메뉴 수   |      5 | 28%    | 더보기를 눌렀을 때 전체 메뉴의 각 점수를 보여줌으로써                                                    |
| 개인 취향·회원      |      4 | 22%    | 사용자의 입맛 취향을 고려하고 비슷한 취향을 가진 사람의 의견 등이 반영되게 기능이 추가 되었으면 좋겠어요 |
| 지도                |      4 | 22%    | 지도버튼을 눌러야 넘어가는게 아니라 배너 전체를 눌러야 지도로 넘어가는거였으면 좋겠고                    |
| 추천 이유·점수 의미 |      3 | 17%    | 음식 찾는 이유도 알려줬으면 좋겠어용                                                                     |
| 분류·정렬           |      3 | 17%    | 메뉴 카테고리가 한식 일식 중식 양식 이렇게도 있으면 좋을 것 같아요                                       |

In [18]:
out = df[["version", "key", "text", "human", "llm", "exact", "evidence", "other"]]
out.to_csv("feedback_llm_results.csv", index=False, encoding="utf-8-sig")
files.download("feedback_llm_results.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 해석 가이드
- **재현율**이 높으면 사람이 묶은 주제를 LLM도 놓치지 않는다는 뜻이다. 우선순위 판단에는 재현율이 더 중요하다 (요청을 놓치면 우선순위가 틀어진다).
- 불일치 중 "사람이 놓친 것"으로 판정된 건은 LLM이 수작업보다 나은 지점이다. 반대 경우는 주제 정의를 다듬어 다시 실행한다.
- 운영 방식: **LLM이 1차 분류 → 사람은 불일치·새 주제 후보만 검토.** 전체를 읽는 대신 검토 대상만 본다.
